In [ ]:
import json
from pathlib import Path
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import numpy as np
import random

In [ ]:
df = pd.read_json("../data/binance_BTCUSDT_depth20_events.jsonl",lines=True)

In [ ]:
# Checking if top 10 of each orderbook data is ordered to begin with
# We can do a lazy check 
# We simply sample 500 datapoints and then check if they are ordered

def is_sorted_side(levels, side):
    # levels: [["price","qty"], ...] or [[price, qty], ...]
    prices = [float(p) for p, _ in levels]
    if side == "bids":
        return all(prices[i] >= prices[i+1] for i in range(len(prices)-1))  # descending
    if side == "asks":
        return all(prices[i] <= prices[i+1] for i in range(len(prices)-1))  # ascending
    raise ValueError("side must be 'bids' or 'asks'")

# quick sample check
idxs = random.sample(range(len(df)), k=min(500, len(df)))
bad_bids = [i for i in idxs if not is_sorted_side(df.iloc[i]["data"]["b"], "bids")]
bad_asks = [i for i in idxs if not is_sorted_side(df.iloc[i]["data"]["a"], "asks")]

print("sample rows checked:", len(idxs))
print("unsorted bids:", len(bad_bids), "example:", bad_bids[:5])
print("unsorted asks:", len(bad_asks), "example:", bad_asks[:5])


In [ ]:
# Pre-allocate arrays
bids_array = np.array([[float(p), float(q)] for orderbook in df["data"] for p, q in orderbook["b"]])
asks_array = np.array([[float(p), float(q)] for orderbook in df["data"] for p, q in orderbook["a"]])

# Depth per row (number of bid/ask levels in each event)
depth_df = pd.DataFrame({
    "bid_depth": df["data"].map(lambda x: len(x.get("b", []))),
    "ask_depth": df["data"].map(lambda x: len(x.get("a", []))),
})

depth_df["min_side_depth"] = depth_df[["bid_depth", "ask_depth"]].min(axis=1)

# 1) Quick summary
print(depth_df.describe())
print("Global min depth:", depth_df["min_side_depth"].min())

# 2) Rows with minimum depth
min_depth = depth_df["min_side_depth"].min()
rows_min = depth_df.index[depth_df["min_side_depth"] == min_depth]
print("Rows with minimum depth:", rows_min.tolist()[:50])  # first 50 if many

# optional: view original rows
# df_min = df.loc[rows_min]
# display(df_min.head())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

depth_df["bid_depth"].hist(ax=ax[0], bins=40)
ax[0].set_title("Bid Depth per Row")
ax[0].set_xlabel("Levels")
ax[0].set_ylabel("Count")

depth_df["ask_depth"].hist(ax=ax[1], bins=40)
ax[1].set_title("Ask Depth per Row")
ax[1].set_xlabel("Levels")

plt.tight_layout()
plt.show()


In [ ]:
# Testing out stuff
# Some simple data cleaning,
# Assign the data type to something and then individually explore what I Can do with it
# Assuming that the orders are all new



test = df.iloc[0,4]
# print(test)
type(test)

# Pulling out orderboook
ask = [[float(p), float(q)] for p, q in test['a']]
bid = [[float(p), float(q)] for p, q in test['b']]

# Calculating best bid and best ask
best_ask_price =min(ask, key= lambda x:x[0] )
best_bid_price=max(bid, key= lambda x:x[0] )

# Calculating spread
spread = best_ask_price[0] - best_bid_price[0]

# Calculating VWPA
def VWPA(bids, asks):
    volume_bid = volume_ask = 0.0
    vol_price_bid = vol_price_ask = 0.0

    n = min(len(bids), len(asks))
    for x in range(n):
        volume_bid += bids[x][1]
        volume_ask += asks[x][1]
        vol_price_bid += bids[x][1] * bids[x][0]
        vol_price_ask += asks[x][1] * asks[x][0]

    return (vol_price_bid / volume_bid, vol_price_ask / volume_ask)



# Checks
print("This is ask")
print(best_ask_price)
print(ask),print('')

print("This is bid")
print(best_bid_price)
print(bid),print(" ")

print("This is the current spread"),print(spread)

print('This is the VWPA')
print(VWPA(bid,ask))

